In [1]:
library(sceasy)
library(data.table)
library(dplyr)

Loading required package: reticulate


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
sce = readRDS("/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/processed/SingleCellExperiment.rds")

In [3]:
args = list()
args$metadata = "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/results/rna/mapping/sample_metadata_after_mapping.txt.gz"
sample_metadata <- fread(args$metadata) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE]

In [11]:
sce = sce[,sample_metadata$cell]

In [ ]:
genes = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/results/genes_scanvi.csv')$name

In [16]:
lost = genes[!genes %in% rownames(sce)]

In [21]:
m = as(matrix(0, nrow = length(lost), ncol = ncol(sce)), "dgCMatrix")

In [25]:
colnames(m) = colnames(sce)
rownames(m) = lost

In [27]:
counts = counts(sce)

In [30]:
counts_new = rbind(counts, m)

In [34]:
counts_new = counts_new[genes,]

In [36]:
sce_new = SingleCellExperiment(list(counts = counts_new), colData = sample_metadata)

In [37]:
sce_new

class: SingleCellExperiment 
dim: 2500 45670 
metadata(0):
assays(1): counts
rownames(2500): Ralb Pax3 ... Gm28578 2610016A17Rik
rowData names(0):
colnames(45670): SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1
  SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1 ...
  SLX-21143_SITTH3_HTJH3DSX2#TTTGTTGGTAAGATCA-1
  SLX-21143_SITTH3_HTJH3DSX2#TTTGTTGGTCATCGGC-1
colData names(17): cell sample ... idx tdTom_corr
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [39]:
sceasy::convertFormat(sce_new, from="sce", 
                      to="anndata",
                      outFile='/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/processed/anndata_scanvi.h5ad')

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:pass_rnaQC, doublet_call”


AnnData object with n_obs × n_vars = 45670 × 2500
    obs: 'cell', 'sample', 'barcode', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'stage', 'tdTom', 'doublet_score', 'celltype.mapped_mnn', 'celltype.score_mnn', 'closest.cell_mnn', 'idx', 'tdTom_corr'
    var: 'name'